# Instructions

(1) In mod08_sustainability_calculations, the energy usage is the product of the number of CPU cores, the number of runtime hours, and the number of kilowatts used per core. Update the function "compute_energy_kwh" to reflect this. _Hint_: You'll want to explore the data created in mod08_generate_emissions_data to learn the appropriate variable names.

(2) After you've updated (and saved) mod08_sustainability_calculations, run the cells in this notebook and answer the three discussion question. 

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from mod08_generate_emissions_data import generate_data
from mod08_sustainability_calculations import (
    compute_energy_kwh,
    compute_emissions,
    emissions_by_region,
    compute_total_runtime
)

### Get the data

In [ ]:
jobs, carbon_intensity, power_per_core_kw, max_cores_per_region = generate_data()

### Inspect the Provided Data

In [ ]:
print("Jobs Dataset:")
print(jobs)

print("\nCarbon Intensity by Region (kg CO2 per kWh):")
for region, intensity in carbon_intensity.items():
    print(f"{region}: {intensity}")

### Compute Energy Usage

In [ ]:
jobs_with_energy = compute_energy_kwh(jobs, power_per_core_kw)

print("Jobs with Energy Computed:")
print(jobs_with_energy)

total_energy = jobs_with_energy["energy_kwh"].sum()
print(f"\nTotal Energy Consumption: {total_energy:.4f} kWh")

### Compute Emissions for Each Region

In [ ]:
region_emissions = emissions_by_region(jobs_with_energy, carbon_intensity)

print("Total Emissions by Region (kg CO2):")
for region, emissions in region_emissions.items():
    print(f"{region}: {emissions:.4f}")

### Visualize Emissions by Region

In [ ]:
plt.figure()
plt.bar(region_emissions.keys(), region_emissions.values())
plt.xlabel("Region")
plt.ylabel("Total Emissions (kg CO2)")
plt.title("Carbon Emissions by Region")
plt.xticks(rotation=45)
plt.show()

### Scenario 1: All Jobs in Coal-Heavy Region

In [ ]:
coal_region = max(carbon_intensity, key=carbon_intensity.get)
all_coal_emissions = compute_emissions(
    jobs_with_energy,
    carbon_intensity[coal_region]
)

all_coal_runtime = compute_total_runtime(jobs_with_energy, max_cores_per_region[coal_region])

print(f"All jobs in {coal_region}: {all_coal_emissions:.4f} kg CO2, Runtime: {all_coal_runtime:.2f} h")

### Scenario 2: All Jobs in Renewable Region

In [ ]:
renewable_region = min(carbon_intensity, key=carbon_intensity.get)
all_renewable_emissions = compute_emissions(
    jobs_with_energy,
    carbon_intensity[renewable_region]
)

all_renewable_runtime = compute_total_runtime(jobs_with_energy, max_cores_per_region[renewable_region])

print(f"All jobs in {renewable_region}: {all_renewable_emissions:.4f} kg CO2, Runtime: {all_renewable_runtime:.2f} h")

### Scenario 3: Assign High CPU Jobs to Renewable Region

In [ ]:
# Decide which jobs to assign to renewable region
median = jobs_with_energy["cpu_cores"].median()

green_jobs = jobs_with_energy[jobs_with_energy["cpu_cores"] >= median]
dirty_jobs = jobs_with_energy[jobs_with_energy["cpu_cores"] < median]

# compute the new total emissions
smart_emissions = compute_emissions(green_jobs, carbon_intensity[renewable_region]) + \
                  compute_emissions(dirty_jobs, carbon_intensity[coal_region])

# compute the new total runtime (with parallelization)
smart_runtime = max(compute_total_runtime(green_jobs, max_cores_per_region[renewable_region]),
                    compute_total_runtime(dirty_jobs, max_cores_per_region[coal_region]))

print(f"High-CPU jobs to {renewable_region}: {smart_emissions:.4f} kg CO2, Runtime: {smart_runtime:.2f} h")

### Strategy Comparison Plot

In [ ]:
# Prepare data
strategy_names = ["All Coal", "All Renewable", "Smart Assignment"]
emission_values = [all_coal_emissions, all_renewable_emissions, smart_emissions]
runtime_values = [all_coal_runtime, all_renewable_runtime, smart_runtime]

# Plot
fig, ax1 = plt.subplots(figsize=(8,5))

# Bar chart for emissions
ax1.bar(strategy_names, emission_values, color='skyblue', alpha=0.7)
ax1.set_ylabel("Total Emissions (kg CO2)", color='blue')
ax1.set_xlabel("Deployment Strategy")
ax1.tick_params(axis='y', labelcolor='blue')
ax1.set_title("Emissions and Total Runtime by Deployment Strategy")

# Line chart for runtime
ax2 = ax1.twinx()
ax2.plot(strategy_names, runtime_values, color='red', marker='o', linewidth=2)
ax2.set_ylabel("Total Runtime (hours)", color='red')
ax2.tick_params(axis='y', labelcolor='red')
ax2.set_ylim(bottom=0)

plt.show()

# Discussion Questions

### Similar to what we did in Scenario 3, let's look at different quantile values to decide what proportion of jobs to assign to renewable energy.

In [ ]:
# Decide which jobs to assign to renewable region
cpu_threshold = jobs_with_energy["cpu_cores"].quantile(
    # enter a choice between 0 and 1 here
    0.75
)

green_jobs = jobs_with_energy[jobs_with_energy["cpu_cores"] >= cpu_threshold]
dirty_jobs = jobs_with_energy[jobs_with_energy["cpu_cores"] < cpu_threshold]

# compute the new total emissions
smart_emissions = compute_emissions(green_jobs, carbon_intensity[renewable_region]) + \
                  compute_emissions(dirty_jobs, carbon_intensity[coal_region])

# compute the new total runtime (with parallelization)
smart_runtime = max(compute_total_runtime(green_jobs, max_cores_per_region[renewable_region]),
                    compute_total_runtime(dirty_jobs, max_cores_per_region[coal_region]))

print(f"High-CPU jobs to {renewable_region}: {smart_emissions:.4f} kg CO2, Runtime: {smart_runtime:.2f} h")

### Comment on what you believe is the most appropriate threshold given the tradeoff between emissions and runtime.

I think a threshold around 0.75 (which gives a cpu_cores cutoff of 8) is the best balance here. At q=0.75 you get emissions of 1.2620 kg CO2 and a runtime of 8.85 hours. Compare that to the extremes: at q=0 (everything on renewables) you get 0.3695 kg CO2 but runtime jumps to 11.95 hours, and at q=0.9 (almost everything on coal) runtime drops to 4.10 hours but emissions spike to 3.8630 kg CO2. So at 0.75 you're only about 0.9 kg CO2 above the absolute minimum emissions, but you shave 3 hours off the runtime compared to going all renewable. Going lower than 0.75, like 0.5 or 0.4, doesn't actually change the emissions or runtime at all because the cpu_cores values are clustered, it still maps to the same split of 9 green and 6 dirty jobs. And going higher to 0.9 saves about 5 more hours of runtime but nearly triples the emissions. So 0.75 feels like the sweet spot where you're keeping emissions pretty low without paying too big of a cost in how long things take.

### In this notebook, we calculate emissions and runtimes as the inputs of our cost-benefit analysis. Name and explain at least one other cost or benefit that is not considered here.

One big cost not accounted for here is the actual dollar cost of running in each region. Renewable energy regions might have higher or lower electricity prices depending on where they are and what infrastructure is set up there. Even if the emissions are way lower in the renewable region, if the price per kWh is significantly higher, it could make it too expensive to run all the heavy jobs there. There's also the cost of data transfer and network latency. If the jobs need to communicate with each other or with a central database, splitting them across two distant regions adds latency and bandwidth costs that could slow things down beyond just the compute time. On the benefit side, there's the reputation or regulatory benefit of being able to say your operations are low carbon, which could matter for companies that have ESG commitments or are in jurisdictions with carbon taxes.

### List at least two different stakeholders that would be affected by a strategic decision regarding the assignment of resources explored in this notebook. In your opinion, which stakeholder's interests are most compelling?

1. The end users or customers who depend on the compute jobs finishing on time. If a company routes everything to the renewable region to minimize emissions but runtime goes from 7 hours to 12 hours, those users are waiting longer for their results, which directly affects their experience.
2. Communities living near the coal heavy data centers. They deal with the air pollution and health effects from the power plants supplying those facilities, so routing more jobs to coal regions means more local harm to those people.
3. The company itself, which has to balance operational costs, speed commitments to customers, and its own sustainability goals or regulatory requirements.

I think the communities near the coal plants have the most compelling case because they're bearing the environmental and health costs of decisions they have no say in. End users might be inconvenienced by slower runtimes but the people living near coal power plants are dealing with real health consequences like respiratory issues and water contamination. The company can adjust its business model but those communities can't easily move away from the pollution.